In [ ]:
# cell 1
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit, ParameterGrid
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)
from imblearn.over_sampling import RandomOverSampler


In [ ]:
# cell 2
# Settings
DATA_DIR = r"..\Data\model_data"
RESULTS_DIR = r"..\Results\Model_selection"

SEED = 10
N_SPLITS = 5

OUTCOME_PREFIXES = [
    "mask",
    "protective",
    "social_avoidance"
]

cv = StratifiedShuffleSplit(
    n_splits=N_SPLITS,
    test_size=1 / N_SPLITS,
    random_state=SEED
)


In [ ]:
# cell 3
# Parameter grid

rf_param_grid = {
    "max_depth": [5, 10, 20],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", "log2", None]
}


In [ ]:
# cell 4
# Helper functions

def load_training_data(prefix):
    X_train = pd.read_csv(f"{DATA_DIR}\\{prefix}_X_train.csv")
    y_train = pd.read_csv(f"{DATA_DIR}\\{prefix}_y_train.csv").values.ravel()

    return X_train, y_train


def prepare_fold_data(X_train, X_val):
    X_train = X_train.copy()
    X_val = X_val.copy()

    bool_cols_train = X_train.select_dtypes(include=["bool"]).columns
    bool_cols_val = X_val.select_dtypes(include=["bool"]).columns

    if len(bool_cols_train) > 0:
        X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)

    if len(bool_cols_val) > 0:
        X_val[bool_cols_val] = X_val[bool_cols_val].astype(int)

    X_train, X_val = X_train.align(
        X_val,
        join="left",
        axis=1,
        fill_value=0
    )

    return X_train, X_val


def summarize_cv_scores(cv_scores):
    summary_df = pd.DataFrame({
        "fold": cv_scores["fold"],
        "precision": cv_scores["precision"],
        "recall": cv_scores["recall"],
        "roc_auc": cv_scores["roc_auc"],
        "accuracy": cv_scores["accuracy"],
        "f1": cv_scores["f1"]
    })

    mean_row = pd.DataFrame([{
        "fold": "mean",
        "precision": summary_df["precision"].mean(),
        "recall": summary_df["recall"].mean(),
        "roc_auc": summary_df["roc_auc"].mean(),
        "accuracy": summary_df["accuracy"].mean(),
        "f1": summary_df["f1"].mean()
    }])

    se_row = pd.DataFrame([{
        "fold": "se",
        "precision": summary_df["precision"].std(ddof=1) / np.sqrt(N_SPLITS),
        "recall": summary_df["recall"].std(ddof=1) / np.sqrt(N_SPLITS),
        "roc_auc": summary_df["roc_auc"].std(ddof=1) / np.sqrt(N_SPLITS),
        "accuracy": summary_df["accuracy"].std(ddof=1) / np.sqrt(N_SPLITS),
        "f1": summary_df["f1"].std(ddof=1) / np.sqrt(N_SPLITS)
    }])

    summary_df = pd.concat(
        [summary_df, mean_row, se_row],
        ignore_index=True
    )

    return summary_df


def get_mean_se(summary_df, metric):
    mean_value = summary_df.loc[
        summary_df["fold"] == "mean",
        metric
    ].values[0]

    se_value = summary_df.loc[
        summary_df["fold"] == "se",
        metric
    ].values[0]

    return mean_value, se_value


def print_mean_se(summary_df, prefix):
    print("\n" + "=" * 60)
    print(f"Random Forest results for: {prefix}")

    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value, se_value = get_mean_se(summary_df, metric)
        print(f"{metric}: {mean_value:.3f} ± {se_value:.3f}")


In [ ]:
# cell 5
# Model evaluation

def evaluate_rf_params(prefix, params, upsample=True):
    X, y = load_training_data(prefix)

    cv_scores = {
        "fold": [],
        "precision": [],
        "recall": [],
        "roc_auc": [],
        "accuracy": [],
        "f1": []
    }

    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):

        X_train_fold = X.iloc[train_idx].copy()
        y_train_fold = y[train_idx].copy()

        X_val_fold = X.iloc[val_idx].copy()
        y_val_fold = y[val_idx].copy()

        if upsample:
            upsampler = RandomOverSampler(random_state=SEED)
            X_train_fold, y_train_fold = upsampler.fit_resample(
                X_train_fold,
                y_train_fold
            )

            if not isinstance(X_train_fold, pd.DataFrame):
                X_train_fold = pd.DataFrame(
                    X_train_fold,
                    columns=X.columns
                )

        X_train_prepared, X_val_prepared = prepare_fold_data(
            X_train_fold,
            X_val_fold
        )

        model = RandomForestClassifier(
            n_estimators=250,
            bootstrap=True,
            random_state=SEED,
            n_jobs=-1,
            **params
        )

        model.fit(X_train_prepared, y_train_fold)

        y_pred = model.predict(X_val_prepared)
        y_prob = model.predict_proba(X_val_prepared)[:, 1]

        cv_scores["fold"].append(fold_idx)
        cv_scores["precision"].append(
            precision_score(y_val_fold, y_pred, zero_division=0)
        )
        cv_scores["recall"].append(
            recall_score(y_val_fold, y_pred, zero_division=0)
        )
        cv_scores["roc_auc"].append(
            roc_auc_score(y_val_fold, y_prob)
        )
        cv_scores["accuracy"].append(
            accuracy_score(y_val_fold, y_pred)
        )
        cv_scores["f1"].append(
            f1_score(y_val_fold, y_pred, zero_division=0)
        )

    summary = {
        "params": params,
        "precision_mean": np.mean(cv_scores["precision"]),
        "recall_mean": np.mean(cv_scores["recall"]),
        "roc_auc_mean": np.mean(cv_scores["roc_auc"]),
        "accuracy_mean": np.mean(cv_scores["accuracy"]),
        "f1_mean": np.mean(cv_scores["f1"])
    }

    return cv_scores, summary

In [ ]:
# cell 6
# Hyperparameter tuning

def tune_random_forest(prefix, upsample=True):
    all_results = []

    total_combinations = len(list(ParameterGrid(rf_param_grid)))
    print("\n" + "=" * 60)
    print(f"Tuning Random Forest for: {prefix}")
    print(f"Total parameter combinations: {total_combinations}")
    print(f"Upsampling: {upsample}")

    for i, params in enumerate(ParameterGrid(rf_param_grid), start=1):
        _, summary = evaluate_rf_params(
            prefix=prefix,
            params=params,
            upsample=upsample
        )

        all_results.append(summary)

        if i % 20 == 0:
            print(f"Finished {i} / {total_combinations} combinations")

    results_df = pd.DataFrame(all_results)

    results_df = results_df.sort_values(
        by=["roc_auc_mean", "f1_mean", "accuracy_mean"],
        ascending=False
    ).reset_index(drop=True)

    best_row = results_df.iloc[0]
    best_params = best_row["params"]

    print(f"\nBest params for {prefix}:")
    print(best_params)
    print("Best mean ROC AUC:", round(best_row["roc_auc_mean"], 3))
    print("Best mean precision:", round(best_row["precision_mean"], 3))
    print("Best mean recall:", round(best_row["recall_mean"], 3))
    print("Best mean accuracy:", round(best_row["accuracy_mean"], 3))
    print("Best mean F1:", round(best_row["f1_mean"], 3))

    results_df.to_csv(
        f"{RESULTS_DIR}\\rf_{prefix}_tuning_results.csv",
        index=False
    )

    return results_df, best_params

In [ ]:
# cell 7
# Cross-validate best RF

def cross_validate_best_rf(prefix, best_params, upsample=True):
    cv_scores, _ = evaluate_rf_params(
        prefix=prefix,
        params=best_params,
        upsample=upsample
    )

    summary_df = summarize_cv_scores(cv_scores)

    summary_df.to_csv(
        f"{RESULTS_DIR}\\rf_{prefix}_cv_summary.csv",
        index=False
    )

    print_mean_se(summary_df, prefix)

    return summary_df

In [ ]:
# cell 8
# Run RF tuning and CV

rf_results = {}
rf_best_params = {}
rf_summaries = {}

for prefix in OUTCOME_PREFIXES:
    results_df, best_params = tune_random_forest(
        prefix=prefix,
        upsample=True
    )

    summary_df = cross_validate_best_rf(
        prefix=prefix,
        best_params=best_params,
        upsample=True
    )

    rf_results[prefix] = results_df
    rf_best_params[prefix] = best_params
    rf_summaries[prefix] = summary_df


Tuning Random Forest for: mask
Total parameter combinations: 162
Upsampling: True
Finished 20 / 162 combinations
Finished 40 / 162 combinations
Finished 60 / 162 combinations
Finished 80 / 162 combinations
Finished 100 / 162 combinations
Finished 120 / 162 combinations
Finished 140 / 162 combinations
Finished 160 / 162 combinations

Best params for mask:
{'criterion': 'entropy', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2}
Best mean ROC AUC: 0.898
Best mean precision: 0.848
Best mean recall: 0.826
Best mean accuracy: 0.827
Best mean F1: 0.837

Random Forest results for: mask
precision: 0.848 ± 0.003
recall: 0.826 ± 0.003
roc_auc: 0.898 ± 0.002
accuracy: 0.827 ± 0.003
f1: 0.837 ± 0.002

Tuning Random Forest for: protective
Total parameter combinations: 162
Upsampling: True
Finished 20 / 162 combinations
Finished 40 / 162 combinations
Finished 60 / 162 combinations
Finished 80 / 162 combinations
Finished 100 / 162 combinations
Finished 120 / 16

In [ ]:
# cell 9
# Build comparison table

rows = []

for prefix, summary_df in rf_summaries.items():
    row = {
        "model": "Random Forest",
        "outcome": prefix,
        "best_params": rf_best_params[prefix],
        "upsample": True
    }

    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value, se_value = get_mean_se(summary_df, metric)

        row[metric] = mean_value
        row[f"{metric}_se"] = se_value
        row[f"{metric}_mean_se"] = f"{mean_value:.3f} ± {se_value:.3f}"

    rows.append(row)

rf_comparison_df = pd.DataFrame(rows)

rf_comparison_df.to_csv(
    f"{RESULTS_DIR}\\rf_cv_comparison.csv",
    index=False
)

rf_comparison_df

,model,outcome,best_params,upsample,precision,precision_se,precision_mean_se,recall,recall_se,recall_mean_se,roc_auc,roc_auc_se,roc_auc_mean_se,accuracy,accuracy_se,accuracy_mean_se,f1,f1_se,f1_mean_se
0,Random Forest,mask,"{'criterion': 'entropy', 'max_depth': 20, 'max...",True,0.848258,0.002733,0.848 ± 0.003,0.826263,0.002601,0.826 ± 0.003,0.898225,0.001912,0.898 ± 0.002,0.826503,0.002628,0.827 ± 0.003,0.837110,0.002450,0.837 ± 0.002
1,Random Forest,protective,"{'criterion': 'entropy', 'max_depth': 20, 'max...",True,0.826428,0.002645,0.826 ± 0.003,0.810086,0.000857,0.810 ± 0.001,0.836120,0.002679,0.836 ± 0.003,0.765400,0.002332,0.765 ± 0.002,0.818169,0.001595,0.818 ± 0.002
2,Random Forest,social_avoidance,"{'criterion': 'gini', 'max_depth': 20, 'max_fe...",True,0.786177,0.002235,0.786 ± 0.002,0.759705,0.002876,0.760 ± 0.003,0.808612,0.003233,0.809 ± 0.003,0.736095,0.002254,0.736 ± 0.002,0.772700,0.002022,0.773 ± 0.002
